In [2]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

from dysts.maps import Henon

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

## Init Reservoir

In [3]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [4]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [5]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [19]:
u_val = 1
u_dist = 400
u_dataset = (1 - t // u_dist % 2) * 2 * u_val - u_val
u_dataset = u_dataset[transient_steps_chaos:]

In [140]:
sin_val = 1
sin_dist = 400
sin_dataset = sin_val * np.sin(2 * np.pi * t / (2 * sin_dist))
sin_dataset = sin_dataset[transient_steps_chaos:]

## Init Funcs

In [6]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [7]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [8]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [9]:
def weight_plot(weights, dims):
    dim_names = ["x", "y", "z"][:dims]
    num_features_per_type = len(weights) // 2
    num_nodes = num_features_per_type // dims

    labels = []
    for n in range(num_nodes):
        for d in range(dims):
            labels.append(f"Node {n + 1} {dim_names[d]} Pos")
            labels.append(f"Node {n + 1} {dim_names[d]} Vel")

    weights_flat = weights.flatten()
    pos_part = weights_flat[:num_features_per_type]
    vel_part = weights_flat[num_features_per_type:]

    combined_weights = np.empty_like(weights_flat)
    combined_weights[0::2] = pos_part
    combined_weights[1::2] = vel_part

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Node State & Dimension",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [10]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [11]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [12]:
@njit(cache=True)
def run_simulation(
    steps, dt, m_inv_diag, c_diag, U, initial_pos, connections_list, k_vals, rest_lens, wall_nodes=[-1]
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + .5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Single

In [155]:
N = 1
dist_between = 2.5

x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1, 2]), N)

nodes_pos = np.column_stack((x, y))

In [156]:
rng_val = 42
rng = np.random.default_rng(rng_val)
sigma = 0

dt = 0.01
input_force = 10
m_val = 0.01
c_val = 0.2
k_wall_val = 10
k_between_val = 8
rest_val = 1.1

target_node_count = 1
N_step = int(N / (target_node_count + 1))
target_nodes = 1 + (np.arange(1, (target_node_count + 1)) * N_step) * 3

starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

mu = np.log(m_val) - (sigma**2 / 2)
m_nodes = rng.lognormal(mean=mu, sigma=sigma, size=num_nodes)
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

mu = np.log(c_val) - (sigma**2 / 2)
c_nodes = rng.lognormal(mean=mu, sigma=sigma, size=num_nodes)
c_diag = np.repeat(c_nodes, dims)

In [157]:
node_ids = np.arange(x.size)

starts = 3 * np.arange(N)
wall_src = np.column_stack([starts, starts + 1]).flatten()
wall_dst = np.column_stack([starts + 1, starts + 2]).flatten()

betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]

src_nodes = np.concatenate([wall_src, between_src])
dst_nodes = np.concatenate([wall_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

rng = np.random.default_rng(rng_val)

mu = np.log(k_wall_val) - (sigma**2 / 2)
k_connection_vals = rng.lognormal(
    mean=mu, sigma=sigma, size=wall_src.shape[0] // 2
).repeat(2)
mu = np.log(k_between_val) - (sigma**2 / 2)
k_between_vals = rng.lognormal(mean=mu, sigma=sigma, size=between_src.shape[0])
k_vals = np.concatenate([k_connection_vals, k_between_vals])

mu = np.log(rest_val) - (sigma**2 / 2)
wall_rest_len_randomness = rng.lognormal(
    mean=mu, sigma=sigma, size=(wall_src.shape[0] + 1) // 2
).repeat(2)[: wall_src.shape[0]]
wall_rest_len_randomness = np.clip(wall_rest_len_randomness, a_min=None, a_max=2)

between_rest_sigma = 0
mu = np.log(rest_val) - (between_rest_sigma**2 / 2)
between_rest_len_randomness = rng.lognormal(
    mean=mu, sigma=between_rest_sigma, size=between_src.shape[0]
)

init_vecs = nodes_pos[connections_list[:, 0]] - nodes_pos[connections_list[:, 1]]
rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))
rest_lens[: wall_src.shape[0]] *= wall_rest_len_randomness
rest_lens[wall_src.shape[0] :] *= between_rest_len_randomness

In [158]:
total_steps = steps + transient_steps_reservoir + tau_steps

U = np.zeros((total_steps, matrix_size))
for i, node_index in enumerate(target_nodes):
    U[:, node_index * dims] = u_dataset

In [159]:
displacement, velocity = run_simulation(
    steps=total_steps,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * input_force,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    wall_nodes=wall_nodes,
)

In [160]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
).show()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=displacement[:, target_nodes[0] * dims],
        y=U[:, target_nodes[0] * dims],
        mode="lines",
        marker=dict(size=1),
    )
)

fig.show()

: 

# Single

In [141]:
N = 1
dist_between = 2.5

x = np.zeros(N * 3)
for i in range(0, N):
    x[i * 3 : (i + 1) * 3] = np.array([0, 0, 0]) + dist_between * i
y = np.tile(np.array([0, 1, 2]), N)

nodes_pos = np.column_stack((x, y))

In [142]:
rng_val = 42
rng = np.random.default_rng(rng_val)
sigma = 0

dt = 0.01
input_force = 10
m_val = 0.01
c_val = 0.2
k_wall_val = 10
k_between_val = 8
rest_val = 1.1

target_node_count = 1
N_step = int(N / (target_node_count + 1))
target_nodes = 1 + (np.arange(1, (target_node_count + 1)) * N_step) * 3

starts = 3 * np.arange(N)
wall_nodes = np.column_stack([starts, starts + 2]).flatten()

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

mu = np.log(m_val) - (sigma**2 / 2)
m_nodes = rng.lognormal(mean=mu, sigma=sigma, size=num_nodes)
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

mu = np.log(c_val) - (sigma**2 / 2)
c_nodes = rng.lognormal(mean=mu, sigma=sigma, size=num_nodes)
c_diag = np.repeat(c_nodes, dims)

In [143]:
node_ids = np.arange(x.size)

starts = 3 * np.arange(N)
wall_src = np.column_stack([starts, starts + 1]).flatten()
wall_dst = np.column_stack([starts + 1, starts + 2]).flatten()

betweens = np.arange(1, node_ids[-1], 3)
between_src = betweens[:-1]
between_dst = betweens[1:]

src_nodes = np.concatenate([wall_src, between_src])
dst_nodes = np.concatenate([wall_dst, between_dst])
connections_list = np.column_stack((src_nodes, dst_nodes))

rng = np.random.default_rng(rng_val)

mu = np.log(k_wall_val) - (sigma**2 / 2)
k_connection_vals = rng.lognormal(
    mean=mu, sigma=sigma, size=wall_src.shape[0] // 2
).repeat(2)
mu = np.log(k_between_val) - (sigma**2 / 2)
k_between_vals = rng.lognormal(mean=mu, sigma=sigma, size=between_src.shape[0])
k_vals = np.concatenate([k_connection_vals, k_between_vals])

mu = np.log(rest_val) - (sigma**2 / 2)
wall_rest_len_randomness = rng.lognormal(
    mean=mu, sigma=sigma, size=(wall_src.shape[0] + 1) // 2
).repeat(2)[: wall_src.shape[0]]
wall_rest_len_randomness = np.clip(wall_rest_len_randomness, a_min=None, a_max=2)

between_rest_sigma = 0
mu = np.log(rest_val) - (between_rest_sigma**2 / 2)
between_rest_len_randomness = rng.lognormal(
    mean=mu, sigma=between_rest_sigma, size=between_src.shape[0]
)

init_vecs = nodes_pos[connections_list[:, 0]] - nodes_pos[connections_list[:, 1]]
rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))
rest_lens[: wall_src.shape[0]] *= wall_rest_len_randomness
rest_lens[wall_src.shape[0] :] *= between_rest_len_randomness

In [144]:
total_steps = steps + transient_steps_reservoir + tau_steps

U = np.zeros((total_steps, matrix_size))
for i, node_index in enumerate(target_nodes):
    U[:, node_index * dims] = sin_dataset

In [145]:
displacement, velocity = run_simulation(
    steps=total_steps,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * input_force,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    wall_nodes=wall_nodes,
)

In [146]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    max_frames=20000,
    steps_jump=1,
).show()

In [147]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=displacement[:, target_nodes[0] * dims],
        y=U[:, target_nodes[0] * dims],
        mode="lines",
        marker=dict(size=1),
    )
)

fig.show()

# Single Hysteron

In [148]:
e_low, e_high = -.4, .6
m1, m2 = 5.0, 5.0
g = 5.0
state = 0

In [ ]:
hysteron_steps = 1000
position = []

for displacement in sin_dataset[:hysteron_steps]:
    if displacement <= e_low:
        state = 0
    elif displacement >= e_high:
        state = 1
    # else keep state

    if state == 1:
        position.append((m1 * displacement) - g)
    elif state == 0:
        position.append(m2 * displacement)

In [153]:
x_vals = sin_dataset[:hysteron_steps]
y_vals = position

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Hysteron Loop",
        line=dict(color="blue", width=2),
        marker=dict(size=6, color="red"),
    )
)

fig.update_layout(
    title="Interactive Hysteresis Loop from Sinusoidal Displacement",
    xaxis_title="Displacement (Sin Dataset)",
    yaxis_title="Position / Force Output",
    template="plotly_white",
    hovermode="closest",
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)

fig.show()

# Single Hysteron matrix

In [ ]:
e_low, e_high = -.4, .6
m1, m2 = 5.0, 5.0
g = 5.0
state = 0

In [154]:
hysteron_steps = 1000
displacements = sin_dataset[:hysteron_steps]
states = np.zeros_like(displacements, dtype=int)

for i, d in enumerate(displacements):
    if d <= e_low:
        current_state = 0
    elif d >= e_high:
        current_state = 1
    states[i] = current_state

m_matrix = np.where(states == 1, m1, m2)
g_matrix = np.where(states == 1, g, 0.0)

position = (m_matrix * displacements) - g_matrix

NameError: name 'current_state' is not defined

In [ ]:
x_vals = sin_dataset[:hysteron_steps]
y_vals = position

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines+markers",
        name="Hysteron Loop",
        line=dict(color="blue", width=2),
        marker=dict(size=6, color="red"),
    )
)

fig.update_layout(
    title="Interactive Hysteresis Loop from Sinusoidal Displacement",
    xaxis_title="Displacement (Sin Dataset)",
    yaxis_title="Position / Force Output",
    template="plotly_white",
    hovermode="closest",
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)

fig.show()